# Google Summer of Code (GSoC) Task - Neural Operators for Fast Simulation of Strong Gravitational Lensing
**Task Specification:** Build an image classifier for strong gravitational lensing images (No substructure, Subhalos, Vortices). Compare a standard CNN to a Neural Operator (FNO) that operates in continuous function space via Spectral Convolutions. Evaluate using ROC curves and AUC scores.

**Why FNO for this task?**
A standard CNN operates on discrete pixels using finite kernels ($3 \times 3$). An FNO maps continuous functions globally via the Convolution Theorem: a spatial convolution is equivalent to multiplication in the frequency domain. FNO transforms the image with an FFT, truncates high-frequency modes, multiplies continuous learnable weights, and inverse-transforms back. It naturally captures the continuous macro-structure of a gravitational lens better than a pixel-by-pixel CNN filter.

# FNO Classifier  — Gravitational Lens Classification
## Specific Test IV · Neural Operators · ML4SCI GSoC 2025

This notebook implements a **Fourier Neural Operator (FNO)** based classifier
for strong gravitational lensing images. The model classifies each image into
one of three physical categories:

| Class | Label | Physical meaning |
|-------|-------|-----------------|
| No substructure | `no` | Clean Einstein ring — no dark matter substructure |
| Subhalo | `sphere` | Ring perturbed by a dark matter subhalo |
| Vortex | `vort` | Ring perturbed by a vortex substructure |

### Why FNO instead of a standard CNN?
A standard CNN uses local 3×3 filters — its receptive field grows slowly with
depth and it needs many layers before it can "see" the full Einstein ring.
An FNO performs a **global spectral convolution via FFT at every layer**, giving
it a full-image receptive field from layer 1. This is physically motivated:
the three classes differ in how substructure perturbs the **global frequency
spectrum** of the ring, not just local pixel patches.

### Evaluation metrics
- ROC curve (one-vs-rest, per class)
- AUC score (per class + macro average)

## 1. Imports & Setup
Standard imports. `tqdm.auto` is used so the progress bar renders
correctly in both Kaggle notebooks and local Jupyter environments.
`sklearn` is used only for evaluation (ROC / AUC) — not for training.

In [1]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
import random
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
import warnings
warnings.filterwarnings('ignore')

## 2. Physics-Aware Augmentation

Standard augmentations designed for natural images (random crops,
colour jitter) are **wrong for gravitational lensing** and would
destroy the ring signal.

We use only augmentations that are consistent with the physics:

| Augmentation | Allowed? | Physical justification |
|---|---|---|
| Rotation 0–360° | ✅ Yes | Einstein rings are rotationally symmetric |
| Horizontal flip | ✅ Yes | Lensing geometry has no preferred orientation |
| Vertical flip | ✅ Yes | Same as above |
| Gaussian noise σ=0.005 | ✅ Yes | Simulates real telescope detector noise |
| Random crop | ❌ No | Would cut off part of the ring |
| Colour jitter | ❌ No | Meaningless for single-channel flux data |
| Elastic distortion | ❌ No | Would warp the ring geometry |

The full 360° rotation range gives approximately **4× more effective
training data** compared to the standard ±15° rotation used in generic
image classifiers.

In [3]:
class LensAugment:
    """
    Physics-aware augmentation for Einstein ring images.

    ALLOWED (preserve ring structure):
      - Rotation by any angle (rings are rotationally symmetric)
      - Horizontal / vertical flip (lens geometry is reflection-symmetric)
      - Mild Gaussian noise (simulates detector noise)

    NOT ALLOWED (would destroy ring signal):
      - Crops (may cut off the ring)
      - ColorJitter (meaningless for 1-channel flux)
      - Elastic distortion (warps the ring geometry)
    """
    def __init__(self, noise_std=0.005):
        self.noise_std = noise_std

    def __call__(self, img):
        # img: tensor (1, H, W), already in [0, 1]
        # Full rotation — valid because ring is rotationally symmetric
        angle = random.uniform(0, 360)
        img = TF.rotate(img, angle, interpolation=TF.InterpolationMode.BILINEAR,
                        fill=0.0)

        # Flip augmentations
        if random.random() > 0.5:
            img = TF.hflip(img)
        if random.random() > 0.5:
            img = TF.vflip(img)
        if self.noise_std > 0:
            noise = torch.randn_like(img) * self.noise_std
            img = (img + noise).clamp(0., 1.)

        return img

## 3. Dataset & Preprocessing

Each `.npy` file is a **2D flux map** — a single-channel grayscale image
representing the brightness of a gravitational lens system as observed
by a telescope.

### Two key preprocessing decisions

**1. Log1p tone-mapping**
The raw images have an extreme dynamic range: the dark void near the
lens galaxy is close to 0, while the bright arc knots can be near 1.0.
Plain min-max normalisation preserves this imbalance. We apply:

$$x' = \frac{\log(1 + s \cdot x)}{\log(1 + s)}, \quad s = 10$$

This compresses the bright peaks without losing the faint ring signal,
making the gradient landscape much smoother for the network.

**2. Resize to 150×150**
The original images may be at different resolutions. We resize all
inputs to 150×150 — large enough to preserve fine arc structure,
small enough to fit in GPU memory at batch size 16.

In [4]:
class LensDataset(Dataset):
    """
    Loads .npy images from dataset/train|val/no|sphere|vort/

    log1p_scale: applies x = log1p(x * scale) / log1p(scale) to compress
                 the extreme bright knots without losing dark-ring contrast.
                 scale=10 is a good default for these images.
    """
    CLASS_MAP = {'no': 0, 'sphere': 1, 'vort': 2}

    def __init__(self, root_dir, split='train', transform=None,
                 img_size=150, log1p_scale=10.0):
        self.samples = []
        self.transform = transform
        self.img_size = img_size
        self.log1p_scale = log1p_scale

        split_dir = os.path.join(root_dir, split)
        for class_name, label in self.CLASS_MAP.items():
            class_dir = os.path.join(split_dir, class_name)
            if not os.path.isdir(class_dir):
                continue
            for fname in sorted(os.listdir(class_dir)):
                if fname.endswith('.npy'):
                    self.samples.append((os.path.join(class_dir, fname), label))

    def __len__(self):
        return len(self.samples)

    def _preprocess(self, img: np.ndarray) -> torch.Tensor:
        """
        img: raw .npy array, any shape
        returns: tensor (1, img_size, img_size) in [0, 1]
        """
        # Collapse to 2D
        if img.ndim == 3:
            img = img[..., 0] if img.shape[-1] <= 4 else img[0]

        # Min-max to [0,1]  (dataset says already done, but re-apply to be safe)
        mn, mx = img.min(), img.max()
        if mx - mn > 1e-8:
            img = (img - mn) / (mx - mn)

        # Optional log1p tone-mapping — compresses bright knots
        if self.log1p_scale and self.log1p_scale > 0:
            s = self.log1p_scale
            img = np.log1p(img * s) / np.log1p(s)

        # To tensor + resize
        t = torch.tensor(img, dtype=torch.float32).unsqueeze(0)  # (1, H, W)
        t = F.interpolate(t.unsqueeze(0), size=(self.img_size, self.img_size),
                          mode='bilinear', align_corners=False).squeeze(0)
        return t

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = np.load(path).astype(np.float32)
        img = self._preprocess(img)
        if self.transform:
            img = self.transform(img)
        return img, label


def get_loaders(data_root, batch_size=32, img_size=150, log1p_scale=10.0):
    train_ds = LensDataset(data_root, 'train',
                           transform=LensAugment(noise_std=0.005),
                           img_size=img_size, log1p_scale=log1p_scale)
    val_ds   = LensDataset(data_root, 'val',
                           transform=None,  # no augmentation at eval
                           img_size=img_size, log1p_scale=log1p_scale)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False,
                              num_workers=2, pin_memory=True)
    print(f"[Dataset] Train: {len(train_ds)} | Val: {len(val_ds)}")
    return train_loader, val_loader


### 4. SpectralConv2d — the core FNO layer

This is what makes the FNO fundamentally different from a CNN.

**What it does:**
1. Applies `rfft2` — transforms the entire feature map to frequency space
2. Keeps only the **16 lowest Fourier modes** in each dimension
3. Multiplies by learnable **complex-valued weights** `R_θ`
4. Applies `irfft2` — transforms back to pixel space

**Why this matters for lensing:**
- Low Fourier modes encode **global ring geometry** — arc position,
  Einstein radius, overall brightness distribution
- High modes encode pixel-level noise — irrelevant to classification
- By truncating at k=16, the network automatically focuses on the
  physically meaningful signal and discards noise

**Key difference from CNN convolution:**
A CNN kernel slides over local patches (3×3 pixels at a time).
`SpectralConv2d` operates on the **entire image at once** through its
frequency representation — every output pixel depends on every input
pixel, from the very first layer.

In [5]:

class SpectralConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, modes1, modes2):
        super().__init__()
        self.in_channels  = in_channels
        self.out_channels = out_channels
        self.modes1 = modes1
        self.modes2 = modes2
        scale = 1 / (in_channels * out_channels)
        self.weights1 = nn.Parameter(
            scale * torch.randn(in_channels, out_channels, modes1, modes2, dtype=torch.cfloat))
        self.weights2 = nn.Parameter(
            scale * torch.randn(in_channels, out_channels, modes1, modes2, dtype=torch.cfloat))

    def compl_mul2d(self, inp, weights):
        return torch.einsum("bixy,ioxy->boxy", inp, weights)

    def forward(self, x):
        B, C, H, W = x.shape
        x_ft = torch.fft.rfft2(x)
        out_ft = torch.zeros(B, self.out_channels, H, x_ft.shape[-1],
                             dtype=torch.cfloat, device=x.device)
        out_ft[:, :, :self.modes1, :self.modes2] = \
            self.compl_mul2d(x_ft[:, :, :self.modes1, :self.modes2], self.weights1)
        out_ft[:, :, -self.modes1:, :self.modes2] = \
            self.compl_mul2d(x_ft[:, :, -self.modes1:, :self.modes2], self.weights2)
        return torch.fft.irfft2(out_ft, s=(H, W))


class FNOBlock2d(nn.Module):
    def __init__(self, channels, modes1, modes2):
        super().__init__()
        self.spectral = SpectralConv2d(channels, channels, modes1, modes2)
        self.bypass   = nn.Conv2d(channels, channels, 1)
        self.norm     = nn.InstanceNorm2d(channels)

    def forward(self, x):
        return F.gelu(self.norm(self.spectral(x) + self.bypass(x)))


class FNOClassifier(nn.Module):
    def __init__(self, in_channels=1, num_classes=3,
                 width=32, modes1=16, modes2=16, depth=4):
        super().__init__()
        self.lift = nn.Conv2d(in_channels, width, 1)
        self.blocks = nn.Sequential(
            *[FNOBlock2d(width, modes1, modes2) for _ in range(depth)]
        )
        self.pool = nn.AdaptiveAvgPool2d(4)
        hidden = width * 4 * 4
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(hidden, 256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.lift(x)
        x = self.blocks(x)
        x = self.pool(x)
        return self.head(x)



### 5. Single epoch — forward + backward pass

`train_one_epoch` runs one full pass over the training set.

**What happens inside each batch:**
1. Move batch to GPU
2. Forward pass through FNO → get logits
3. Compute CrossEntropyLoss
4. Backward pass → compute gradients
5. **Clip gradients at norm 1.0** — this is important for FNO because
   the complex-valued spectral weights can receive large gradients early
   in training, causing instability without clipping
6. AdamW update → adjust weights
7. Accumulate loss and correct predictions for epoch-level reporting

Returns `(avg_loss, accuracy)` over the full epoch.
### ROC curve and AUC

`plot_roc` computes and plots the ROC curve using a **one-vs-rest**
strategy for each of the three classes.

**How one-vs-rest works:**
For each class (e.g. "Subhalo"), we treat it as a binary problem:
- Positive = this image IS a subhalo
- Negative = this image is NOT a subhalo (could be no-sub or vortex)

We then sweep the classification threshold from 0 to 1 and compute
True Positive Rate and False Positive Rate at each threshold.
This gives one ROC curve per class.

**AUC interpretation:**
- AUC = 1.0 → perfect classifier
- AUC = 0.5 → random guessing (the dashed diagonal line)
- Our result: **Macro-AUC = 0.960**

| Class | AUC | What it means |
|-------|-----|---------------|
| No substructure | 0.977 | Easiest — clean ring has a very distinct spectral signature |
| Vortex | 0.961 | Rotational pattern is a learnable global frequency shift |
| Subhalo | 0.943 | Hardest — localised point perturbation, harder for FFT-based model |

The best model weights (`best_fno.pth`) are reloaded before computing
the final ROC to ensure evaluation is always on the best checkpoint,
not the last epoch.

In [6]:
def train_one_epoch(model, loader, optimizer, criterion, device, epoch, epochs):
    model.train()
    total_loss, correct, n = 0., 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * len(y)
        correct    += (logits.argmax(1) == y).sum().item()
        n          += len(y)
    return total_loss / n, correct / n


@torch.no_grad()
def evaluate(model, loader, criterion, device, epoch=None, epochs=None):
    model.eval()
    total_loss, correct, n = 0., 0, 0
    all_probs, all_labels = [], []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss   = criterion(logits, y)
        probs  = F.softmax(logits, dim=-1)
        total_loss += loss.item() * len(y)
        correct    += (logits.argmax(1) == y).sum().item()
        n          += len(y)
        all_probs.append(probs.cpu().numpy())
        all_labels.append(y.cpu().numpy())
    return total_loss/n, correct/n, np.concatenate(all_probs), np.concatenate(all_labels)


CLASS_NAMES = ['No substructure', 'Subhalo', 'Vortex']

def plot_roc(probs, labels, output_dir, title_prefix='FNO'):
    n_classes = probs.shape[1]
    labels_bin = label_binarize(labels, classes=list(range(n_classes)))
    colors = ['#e41a1c', '#377eb8', '#4daf4a']
    fig, ax = plt.subplots(figsize=(8, 6))
    macro_auc = []
    for i in range(n_classes):
        fpr, tpr, _ = roc_curve(labels_bin[:, i], probs[:, i])
        roc_auc = auc(fpr, tpr)
        macro_auc.append(roc_auc)
        ax.plot(fpr, tpr, color=colors[i], lw=2,
                label=f'{CLASS_NAMES[i]} (AUC = {roc_auc:.3f})')
    ax.plot([0,1],[0,1],'k--',lw=1)
    ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
    ax.set_title(f'{title_prefix} — ROC | Macro-AUC = {np.mean(macro_auc):.3f}')
    ax.legend(loc='lower right'); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'roc_curves.png'), dpi=150)
    plt.close()
    print(f"\n[AUC] " + " | ".join(f"{n}: {a:.4f}" for n, a in zip(CLASS_NAMES, macro_auc)))
    print(f"[AUC] Macro: {np.mean(macro_auc):.4f}")
    return macro_auc


def train(data_root, output_dir='./results', epochs=30,
          batch_size=16, lr=1e-3, img_size=150,
          width=32, modes=16, depth=4,
          log1p_scale=10.0, device=None):

    os.makedirs(output_dir, exist_ok=True)
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"[Device] {device}")

    train_loader, val_loader = get_loaders(data_root, batch_size, img_size, log1p_scale)
    model     = FNOClassifier(in_channels=1, num_classes=3,
                              width=width, modes1=modes, modes2=modes, depth=depth).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    print(f"[Model] FNO2d | Params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
    print(f"[Config] img_size={img_size}, modes={modes}, width={width}, depth={depth}, log1p_scale={log1p_scale}")

    best_val_acc = 0.
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    from tqdm.auto import tqdm
    epoch_bar = tqdm(range(1, epochs + 1), desc="Training", ncols=90, unit='epoch')

    for epoch in epoch_bar:
        tr_loss, tr_acc = train_one_epoch(
            model, train_loader, optimizer, criterion, device, epoch, epochs)
        val_loss, val_acc, val_probs, val_labels = evaluate(
            model, val_loader, criterion, device, epoch, epochs)
        scheduler.step()
        history['train_loss'].append(tr_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(tr_acc)
        history['val_acc'].append(val_acc)

        is_best = val_acc > best_val_acc
        if is_best:
            best_val_acc = val_acc
            torch.save(model.state_dict(), os.path.join(output_dir, 'best_fno.pth'))

        epoch_bar.set_postfix(
            tr_acc=f"{tr_acc:.4f}",
            val_acc=f"{val_acc:.4f}",
            best=f"{best_val_acc:.4f}"
        )
        # Print a clean persistent line every epoch (visible after bar scrolls)
        star = " *best*" if is_best else ""
        print(f"Epoch {epoch:03d}/{epochs} | "
              f"Train loss={tr_loss:.4f} acc={tr_acc:.4f} | "
              f"Val loss={val_loss:.4f} acc={val_acc:.4f}{star}")

    model.load_state_dict(torch.load(os.path.join(output_dir, 'best_fno.pth'), map_location=device))
    _, _, val_probs, val_labels = evaluate(model, val_loader, criterion, device)
    plot_roc(val_probs, val_labels, output_dir)
    print(f"\n[Done] Best Val Acc: {best_val_acc:.4f}")
    return model, history, val_probs, val_labels


## 5. Training Loop

### Optimiser & schedule
- **AdamW** with `lr=1e-3`, `weight_decay=1e-4`
- **CosineAnnealingLR** over 50 epochs — starts at lr=1e-3, smoothly
  decays to near zero, preventing the sharp accuracy drop that
  step-decay schedules cause
- **Gradient clipping** at norm 1.0 — stabilises training when the
  spectral convolution weights receive large gradients early on

### Loss function
**CrossEntropyLoss** — standard for multi-class classification.
No label smoothing is used since the class boundaries in this dataset
are physically well-defined.

### Progress tracking
`tqdm.auto` prints a live progress bar with train loss, train accuracy,
val loss, val accuracy, and best-so-far accuracy after each epoch.
The best model (by val accuracy) is saved automatically to `best_fno.pth`.

In [7]:
train(
    data_root   = '/kaggle/input/datasets/busyfellow/dataset-zip/dataset',
    output_dir  = './results',
    epochs      = 30,
    batch_size  = 16,
    img_size    = 150,
    modes       = 16,
    width       = 32,
    depth       = 4,
    log1p_scale = 10.0,
)

[Device] cuda
[Dataset] Train: 30000 | Val: 7500
[Model] FNO2d | Params: 2,233,539
[Config] img_size=150, modes=16, width=32, depth=4, log1p_scale=10.0


Training:   0%|                                                 | 0/30 [00:00<?, ?epoch/s]

Epoch 001/30 | Train loss=1.1029 acc=0.3311 | Val loss=1.0984 acc=0.3473 *best*
Epoch 002/30 | Train loss=1.0951 acc=0.3563 | Val loss=1.0755 acc=0.4104 *best*
Epoch 003/30 | Train loss=1.0415 acc=0.4353 | Val loss=1.0059 acc=0.4565 *best*
Epoch 004/30 | Train loss=0.9823 acc=0.4888 | Val loss=0.9366 acc=0.5219 *best*
Epoch 005/30 | Train loss=0.9436 acc=0.5134 | Val loss=0.9997 acc=0.4732
Epoch 006/30 | Train loss=0.8948 acc=0.5458 | Val loss=0.8601 acc=0.5589 *best*
Epoch 007/30 | Train loss=0.8521 acc=0.5778 | Val loss=0.8084 acc=0.6217 *best*
Epoch 008/30 | Train loss=0.8138 acc=0.6079 | Val loss=0.7738 acc=0.6300 *best*
Epoch 009/30 | Train loss=0.7729 acc=0.6367 | Val loss=0.7307 acc=0.6684 *best*
Epoch 010/30 | Train loss=0.7325 acc=0.6656 | Val loss=0.6711 acc=0.6923 *best*
Epoch 011/30 | Train loss=0.6906 acc=0.6933 | Val loss=0.8778 acc=0.6256
Epoch 012/30 | Train loss=0.6533 acc=0.7095 | Val loss=0.5887 acc=0.7473 *best*
Epoch 013/30 | Train loss=0.6154 acc=0.7361 | Val loss

(FNOClassifier(
   (lift): Conv2d(1, 32, kernel_size=(1, 1), stride=(1, 1))
   (blocks): Sequential(
     (0): FNOBlock2d(
       (spectral): SpectralConv2d()
       (bypass): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1))
       (norm): InstanceNorm2d(32, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
     )
     (1): FNOBlock2d(
       (spectral): SpectralConv2d()
       (bypass): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1))
       (norm): InstanceNorm2d(32, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
     )
     (2): FNOBlock2d(
       (spectral): SpectralConv2d()
       (bypass): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1))
       (norm): InstanceNorm2d(32, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
     )
     (3): FNOBlock2d(
       (spectral): SpectralConv2d()
       (bypass): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1))
       (norm): InstanceNorm2d(32, eps=1e-05, momentum=0.1, affine=False, track